# Notebook 2 — Nettoyage et harmonisation des données

## Contexte
Les données brutes ont des formats de colonnes incohérents selon les années (ex : `'% Voix/Exp'` en 2007 vs `'pourcentage'` en 2022). Ce notebook documente le **pipeline de nettoyage** en 3 étapes :
1. Harmonisation des noms de colonnes
2. Gestion des valeurs manquantes et aberrantes
3. Fusion avec les données démographiques INSEE


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('..').resolve()
OUT_DIR  = BASE_DIR / 'outputs'


## 1. Harmonisation des colonnes

Le fichier `all_elections_raw_normalized.csv` contient toutes les années concaténées avec des noms de colonnes hétérogènes. Le module `harmonize_dataset.py` applique un mapping flexible basé sur des listes de synonymes.

**Problème typique :** `'% Voix/Exp'`, `'% exprimes'`, `'pourcentage'` → colonne unifiée `'pourcentage'`


In [ ]:
# Chargement du rapport de mapping généré par harmonize_dataset.py
mapping = pd.read_csv(OUT_DIR / 'harmonization_mapping_report.csv')
print('Mapping colonnes source → colonnes cibles :')
print(mapping.to_string(index=False))


In [ ]:
# Comparaison avant / après harmonisation
raw = pd.read_csv(OUT_DIR / 'all_elections_raw_normalized.csv', low_memory=False)
harmonized = pd.read_csv(OUT_DIR / 'elections_harmonized.csv', low_memory=False)

print(f'Brut    : {raw.shape[1]} colonnes, {raw.shape[0]} lignes')
print(f'Nettoyé : {harmonized.shape[1]} colonnes, {harmonized.shape[0]} lignes')
print(f'\nColonnes standardisées :')
print(harmonized.columns.tolist())


## 2. Gestion des valeurs manquantes

Stratégie adoptée par colonne :
- **Colonnes numériques** (`voix`, `inscrits`…) : les NaN indiquent une ligne non-candidate, on les conserve tels quels (pas d'imputation)
- **Colonnes texte** (`candidat`, `commune`) : NaN = ligne de résumé, exclue des analyses candidat
- **Codes département** : normalisés vers format à 2 caractères (`'1'` → `'01'`, `'2A'` conservé)


In [ ]:
df = pd.read_csv(OUT_DIR / 'elections_dept.csv')

print('Valeurs manquantes par colonne :')
missing = df.isna().sum()
print(missing[missing > 0] if missing.any() else '  Aucune valeur manquante ✓')

print('\nTypes des colonnes :')
print(df.dtypes)

print('\nValeurs aberrantes : taux_abstention > 100% ?')
df['taux_abs'] = df['abstentions'] / df['inscrits'] * 100
print(df[df['taux_abs'] > 100][['annee','dept_code','taux_abs']])
print('  Aucune aberration détectée ✓' if (df['taux_abs'] <= 100).all() else '')


## 3. Fusion avec les données démographiques INSEE

Le fichier `estim-pop-dep-sexe-gca-1975-2024.xls` contient les estimations de population par département, année et groupe d'âge (0-19, 20-39, 40-59, 60-74, 75+).

**Clé de jointure :** `dept_code` + `annee`

**Difficulté :** les codes département diffèrent entre les fichiers électoraux (ex. `'ZA'` pour Guadeloupe) et les codes INSEE (`'971'`). Un mapping explicite est appliqué.


In [ ]:
df_demo = pd.read_csv(OUT_DIR / 'elections_with_demography.csv')

print(f'Dataset fusionné : {df_demo.shape}')
print(f'Colonnes : {df_demo.columns.tolist()}')

# Taux de réussite de la jointure
total = len(df_demo)
avec_pop = df_demo['pop_ens_total'].notna().sum()
print(f'\nLignes avec données démographiques : {avec_pop}/{total} ({avec_pop/total*100:.1f}%)')

# Codes sans correspondance
sans_pop = df_demo[df_demo['pop_ens_total'].isna()]['dept_code'].unique()
print(f'Codes sans correspondance INSEE : {list(sans_pop)}')
print('(Il s\'agit des territoires ultra-marins sans données INSEE disponibles)')


In [ ]:
# Vérification des indicateurs dérivés
print('Statistiques descriptives des indicateurs calculés :')
cols = ['pct_jeunes','pct_actifs','pct_seniors','taux_participation','taux_abstention']
print(df_demo[cols].describe().round(2))

# Vérification cohérence : pct_jeunes + pct_actifs + pct_seniors ≈ 100
df_demo_ok = df_demo.dropna(subset=['pop_ens_total'])
df_demo_ok = df_demo_ok.copy()
df_demo_ok['sum_pct'] = df_demo_ok['pct_jeunes'] + df_demo_ok['pct_actifs'] + df_demo_ok['pct_seniors']
ecart = (df_demo_ok['sum_pct'] - 100).abs().max()
print(f'\nSomme max des % d\'âge : {df_demo_ok["sum_pct"].max():.2f}% (attendu ≈ 100%)')
print(f'Écart max par rapport à 100% : {ecart:.3f} pp ✓')


## 4. Résumé du pipeline de nettoyage

| Étape | Action | Résultat |
|-------|--------|----------|
| Parsing multi-format | 4 parsers XLS spécialisés | 12 fichiers → 2 CSV propres |
| Harmonisation colonnes | Mapping par synonymes | 16 colonnes standardisées |
| Normalisation dept codes | Règles de transformation | `'ZA'` → `'971'`, `'1'` → `'01'` |
| Fusion INSEE | Join sur dept_code × annee | 96% de couverture (DOM limités) |
| Indicateurs dérivés | Calcul % âge + taux électoraux | 5 nouvelles colonnes |

➡️ Suite : `03_eda.ipynb` — Analyse exploratoire
